상태: 초안

* 학년:
* 반:
* 번호:
* 이름:

# 오늘 급식 뭐야? — 활동지

**선생님 설명을 들으며 빈 칸에 코드를 직접 입력**하고, 위에서부터 차례대로 실행하세요.

## 1. 급식표를 프로그램이 읽으려면

사람은 학교 홈페이지에서 급식표를 **눈으로** 읽습니다. 프로그램은 화면 대신 **API**라는 창구에 요청을 보내고, 정리된 데이터를 받습니다. 전국 학교의 급식, 시간표, 학사일정은 교육부의 **나이스 교육정보 개방 포털**에서 API로 공개하고 있습니다.

| 홈페이지에서 | 코드에서 |
|---|---|
| 주소창에 주소 입력 | `requests.get(주소, params=...)` |
| 날짜 고르기 | `params`의 `MLSV_YMD` |
| 화면에 나온 급식표 | `response.json()`으로 받은 데이터 |
| 메뉴 찾아 읽기 | `data["mealServiceDietInfo"][1]["row"][0]["DDISH_NM"]` |

### 알아 둘 말

- **API**: 프로그램이 다른 프로그램의 기능이나 데이터를 쓰는 창구입니다.
- **API 키**: 누가 보낸 요청인지 알리는 값으로, 비밀번호처럼 다룹니다. 오늘은 선생님이 알려 주는 **수업용 키**를 씁니다. 실제 키는 선생님 서버에만 있습니다.
- **JSON**: 프로그램끼리 데이터를 주고받는 형식입니다. 파이썬의 딕셔너리, 리스트와 거의 같게 생겼습니다.

오늘은 학생들의 요청을 **선생님 서버**가 모아 나이스로 전달합니다.

```
내 코드 → 선생님 서버 → 나이스 API
```

## 2. 함수로 나누어 만들기

오늘 만드는 프로그램은 **작은 함수 여러 개**를 조립한 것입니다. 함수 하나는 **한 가지 일**만 합니다.

| 함수 | 하는 일 | 받는 것 → 돌려주는 것 |
|---|---|---|
| `get_meal` | 나이스에서 급식 가져오기 | 날짜 → 급식 데이터 |
| `clean_menu` | 메뉴 글자 정리하기 | 지저분한 글자 → 메뉴 리스트 |
| `has_food` | 먹고 싶은 메뉴가 있는지 확인하기 | 메뉴 리스트, 음식 → `True`/`False` |
| `get_calories` | 열량을 숫자로 바꾸기 | `"674.4 Kcal"` → `674.4` |
| `show_meal` | 위 함수들을 조립해 보여 주기 | 날짜 → 화면 출력 |
| `next_food` | 다음에 그 메뉴가 나오는 날 찾기 | 음식, 시작 날짜 → 날짜 |

### `print`와 `return`은 다릅니다

- `print`는 **사람에게 보여 줍니다.** 다른 코드가 그 값을 쓸 수 없습니다.
- `return`은 **코드에게 돌려줍니다.** 돌려받은 값을 다음 함수에 넘길 수 있습니다.

그래서 값을 만드는 함수는 `return`을 쓰고, 화면에 보여 주는 일은 마지막 함수에 맡깁니다.

## 3. 단계별로 따라 하기

### 단계 0: 준비

아래 코드를 실행하세요. (라이브러리 설치)

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('requests') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)

### 단계 1: 접속 설정

`선생님IP`와 `수업용키`를 선생님이 알려 주는 값으로 바꾸어 입력하세요. `DAY`에는 선생님이 알려 주는 **급식이 있는 날**을 `연월일` 8자리로 넣으세요.

### 단계 2: 접속 테스트

`status`가 `ok`이고 우리 학교 이름이 나오면 접속에 성공한 것입니다.

### 단계 3: 요청 보내기

`params`에 원하는 내용을 적어 보냅니다. `MLSV_YMD`는 급식 날짜, `MMEAL_SC_CODE`는 끼니(1 조식, 2 중식, 3 석식)입니다. 출력에서 메뉴가 어디 있는지 찾아보세요.

### 단계 4: 응답에서 값 꺼내기

JSON은 딕셔너리와 리스트가 겹겹이 들어 있습니다. 안쪽으로 한 겹씩 들어가 봅니다. `[0]`에는 요청 결과 정보가, `[1]`에는 실제 데이터가 있습니다.

### 단계 5: 급식이 없는 날

일요일을 물어보면 어떻게 될까요? 응답의 모양이 단계 3과 어떻게 다른지 확인하세요. 프로그램은 **잘 될 때뿐 아니라 안 될 때도** 생각해야 합니다.

### 단계 6: `get_meal` 함수 만들기

단계 3~5를 함수 하나로 묶습니다. 이제 날짜만 넘기면 급식을 받습니다. `meal_code="2"`는 **기본값**입니다. 값을 넘기지 않으면 중식을 가져옵니다. 급식이 없으면 `None`을 돌려줍니다.

### 단계 7: `clean_menu` 함수 만들기

`DDISH_NM`에는 줄바꿈 표시 `<br/>`와 알레르기 번호가 섞여 있습니다. 메뉴 이름만 담은 리스트로 바꿉니다. 이 함수는 인터넷이 필요 없어서, 예시 글자로 바로 시험할 수 있습니다.

- `split("<br/>")`: 글자를 `<br/>` 기준으로 나눕니다.
- `split(" (")[0]`: `돈까스 (1.5)`에서 괄호 앞만 남깁니다.
- `rstrip("0123456789. ")`: `김치9.`에서 끝의 숫자와 점, 공백을 지웁니다.

### 단계 8: `has_food` 함수 만들기

메뉴에 먹고 싶은 음식이 있는지 확인합니다. `in`을 쓰면 `"돈까스"`로 `"치즈돈까스"`도 찾을 수 있습니다. 찾자마자 `return`하면 함수가 바로 끝납니다.

### 단계 9: `get_calories` 함수 직접 만들기 ✏️

이번에는 **선생님 설명 없이 직접** 만듭니다. `"674.4 Kcal"` 같은 글자를 받아 숫자 `674.4`를 돌려주는 함수를 완성하세요.

**힌트**: `split()`으로 공백을 기준으로 나누고, `float()`로 숫자로 바꿉니다.

아래 코드의 결과가 `674.4`, `True`가 되면 성공입니다.

```python
print(get_calories("674.4 Kcal"))
print(get_calories("1066.8 Kcal") > 1000)
```

### 단계 10: 함수 조립하기

지금까지 만든 함수를 호출하기만 하면 됩니다. 새로운 기능을 **다시 쓰지 않고 조립**했다는 점에 주목하세요.

### 단계 11: 일주일 치 모으기

날짜 계산은 파이썬의 `datetime`이 대신합니다. `next_day`로 다음 날을 구하면 `for` 문으로 여러 날을 돌 수 있습니다. 연월 경계도 알아서 처리됩니다.

### 단계 12: 다음 치킨은 언제?

`get_meal`, `clean_menu`, `has_food`, `next_day`를 모두 조립합니다. `food`를 내가 좋아하는 음식으로 바꾸어 보세요.

월말이나 학기 말이면 앞으로의 식단이 아직 올라오지 않았을 수 있습니다.

## 4. 이런 것도 가능해요

같은 방법으로 **내가 쓸 도구**를 만들어 보세요. 아래에서 하나를 고르거나, 나만의 아이디어로 만듭니다. 어느 것이든 **함수로 나누고**, 값을 만드는 함수는 `return`을 쓰세요.

| 만들 것 | 사용할 API | 함수 예시 |
|---|---|---|
| **내일 뭐 챙겨?** 내일 시간표 보고 준비물 알려 주기 | 선생님 서버 `hisTimetable` | `get_timetable(date, grade, class_nm)` |
| **시험까지 며칠?** 학사일정에서 행사 찾아 D-day 계산하기 | 선생님 서버 `SchoolSchedule` | `next_event(name)`, `d_day(date)` |
| **오늘 우산 챙겨?** 강수 확률로 조언하기 | Open-Meteo (키 없음) | `get_rain(lat, lon)`, `rain_advice(percent)` |
| **오늘 체육 해도 돼?** 미세먼지 수치를 등급으로 바꾸기 | Open-Meteo (키 없음) | `get_dust(lat, lon)`, `dust_grade(pm10)` |
| **이번 달 급식 분석** 평균 열량, 자주 나온 메뉴 순위 | 오늘 만든 함수 | `month_menu(start)`, `top_foods(menus)` |
| **아침 브리핑** 위 함수를 합쳐 하루 정보 한 번에 보기 | 내가 만든 함수 전부 | `morning_brief(date)` |

### 사용할 API

**시간표** (학교 코드는 서버가 채워 줍니다.)

```python
params = {"KEY": KEY, "Type": "json", "AY": "2026", "GRADE": "1", "CLASS_NM": "1", "ALL_TI_YMD": DAY}
data = requests.get(BASE_URL + "/hub/hisTimetable", params=params).json()
# data["hisTimetable"][1]["row"] 안의 각 줄: PERIO(교시), ITRT_CNTNT(과목)
```

**학사일정**

```python
params = {"KEY": KEY, "Type": "json", "AA_FROM_YMD": "20261101", "AA_TO_YMD": "20261231"}
data = requests.get(BASE_URL + "/hub/SchoolSchedule", params=params).json()
# data["SchoolSchedule"][1]["row"] 안의 각 줄: AA_YMD(날짜), EVENT_NM(행사 이름)
```

**날씨** (선생님 서버를 거치지 않고 바로 요청합니다. 위도·경도는 지도에서 찾아 넣으세요.)

```python
params = {"latitude": 37.57, "longitude": 126.98, "daily": "precipitation_probability_max",
          "timezone": "Asia/Seoul", "forecast_days": 1}
data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()
# data["daily"]["precipitation_probability_max"][0]: 오늘의 최대 강수 확률(%)
```

**미세먼지** (전 세계 예측 모델 값이라 실제 측정값과 다를 수 있습니다.)

```python
params = {"latitude": 37.57, "longitude": 126.98, "current": "pm10,pm2_5", "timezone": "Asia/Seoul"}
data = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality", params=params).json()
# data["current"]["pm10"]: 미세먼지 농도
```

**알아 두기**

- 응답이 어떻게 생겼는지 모르겠으면 먼저 `print(data)`로 출력해 보세요.
- 수업 서버에서는 한 사람이 1분에 120번까지 요청할 수 있습니다.
- 이름, 학번, 연락처, 건강 정보 같은 개인정보는 코드에 쓰지 않습니다.

### 계획 세우기

| 항목 | 내 계획 |
|---|---|
| 만들 도구 | |
| 사용할 API | |
| 만들 함수 3개 이상 (이름, 받는 것, 돌려주는 것) | |

In [ ]:
# 여기에 코드를 작성하세요.

### 되돌아보기

1. 함수로 나누어서 좋았던 점은 무엇인가요? 다시 쓴 함수가 있었나요?
2. 데이터가 없거나 예상과 다른 모양일 때 프로그램이 어떻게 동작하게 했나요?